# Chapter 1 — Tensors & Shape Algebra (Practice)

Work through these exercises **after reading** `notes/ch01-tensors-and-shape-algebra.md`.

Each exercise states the *decision you're practicing*, gives a stub cell to fill in, and is followed by a pre-written **verification cell** — run it to grade yourself. Hand-write your answers in your working copy under `solutions/` (this `template/` copy stays pristine), and don't peek at `solved/` until the verification passes or you're genuinely stuck.

In [1]:
# ============================================================
# TOPIC: Tensors & shape algebra — storage, strides, dtypes, broadcasting
# MATH:  position(i_0..i_{n-1}) = offset + sum_k i_k * stride_k
# REF:   B00 ch01 notes — tensors-and-shape-algebra
# ============================================================

# --- Imports ---
import numpy as np
import torch
import torch.nn as nn

# --- Reproducibility & device ---
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"torch version : {torch.__version__}")
print(f"device        : {device}  (every exercise here runs fine on CPU)")

torch version : 2.13.0
device        : cpu  (every exercise here runs fine on CPU)


# Self code from notes

## section 3

In [ ]:
# copy vs share
np_array = np.array([1,2,3,4,5])
tensor_copy = torch.tensor(np_array)
tensor_shared = torch.from_numpy(np_array)

# mutate the original numpy array
np_array[0] = 100

print(f"original numpy array: {np_array}")
print(f"tensor_copy : {tensor_copy}")
print(f"tensor_shared : {tensor_shared}")

original numpy array: [100   2   3   4   5]
tensor_copy : tensor([1, 2, 3, 4, 5])
tensor_shared : tensor([100,   2,   3,   4,   5])


In [ ]:
# fresh allocation and ranges
print(f"torch zeros : {torch.zeros(2, 3)}")
print(f"torch.arange(0, 10, 2) : {torch.arange(0, 10, 2)}")
print(f"torch.empty(2, 2) : ", torch.empty(2, 2))

torch zeros : tensor([[0., 0., 0.],
        [0., 0., 0.]])
torch.arange(0, 10, 2) : tensor([0, 2, 4, 6, 8])
torch.empty(2, 2) - notice the garbage :  tensor([[0., 0.],
        [0., 0.]])


In [5]:
# understanding *_like
blueprint_tensor = torch.tensor([[1.5, 2.5], [3.5, 4.5]], dtype=torch.float64)

# create a zeros tensor based on the blueprint
matching_zeros = torch.zeros_like(blueprint_tensor)

print(f"Blueprint Tensor:\n{blueprint_tensor}")
print(f"Blueprint Dtype: {blueprint_tensor.dtype}\n")
print(f"Zeros_like Tensor:\n{matching_zeros}")
print(f"Matching Dtype: {matching_zeros.dtype}")

Blueprint Tensor:
tensor([[1.5000, 2.5000],
        [3.5000, 4.5000]], dtype=torch.float64)
Blueprint Dtype: torch.float64

Zeros_like Tensor:
tensor([[0., 0.],
        [0., 0.]], dtype=torch.float64)
Matching Dtype: torch.float64


## section 4

In [6]:
# the integer vs float rule
dummy_embedding = nn.Embedding(num_embeddings=10, embedding_dim=4)

# correct: using int64 (long) for token IDs
good_tokens = torch.tensor([1, 2, 3], dtype=torch.int64)
print("success Integer tokens worked: ", dummy_embedding(good_tokens))

success Integer tokens worked:  tensor([[ 0.6784, -1.2345, -0.0431, -1.6047],
        [-0.7521,  1.6487, -0.3925, -1.4036],
        [-0.7279, -0.5594, -0.7688,  0.7624]], grad_fn=<EmbeddingBackward0>)


In [7]:
# deliberate Error: Trying to pss floats
bad_tokens = torch.tensor([1.0, 2.0, 3.0], dtype=torch.float32)
try:
    dummy_embedding(bad_tokens)
except Exception as e:
    print(f"\nCaught Expected Crash: {e}")


Caught Expected Crash: Expected tensor for argument #1 'indices' to have one of the following scalar types: Long, Int; but got torch.FloatTensor instead (while checking arguments for embedding)


In [8]:
# Float16 vs Bfloat16
# simulating a large attention logit
large_value = torch.tensor([70000.0], dtype=torch.float32)

fp16_val = large_value.to(torch.float16)
bf16_val = large_value.to(torch.bfloat16)

print(f"Original (fp32): {large_value.item()}")
print(f"To fp16 (Oops!): {fp16_val.item()} <- Overflowed to Infinity because max is 65504")
print(f"To bf16 (Safe): {bf16_val.item()} <- Survived (but slightly rounded)")

Original (fp32): 70000.0
To fp16 (Oops!): inf <- Overflowed to Infinity because max is 65504
To bf16 (Safe): 70144.0 <- Survived (but slightly rounded)


In [12]:
# The device trap
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")    # for mac users
    print("mps detected")
else:
    device = torch.device("cpu")
    print("no GPU detected, pretending CPU is GPU for this example")

my_tensor = torch.tensor([1, 2, 3])

# the mistake
my_tensor.to(device)
print(f"Device after mistake: {my_tensor.device} <- still on CPU")

# the fix
my_tensor = my_tensor.to(device)
print(f"Device after fix : {my_tensor.device} <- successfully moved")

mps detected
Device after mistake: cpu <- still on CPU
Device after fix : mps:0 <- successfully moved


## section 5

In [13]:
# views (slicing) mutate the original
# create a 3x4 tensor
base_tensor = torch.arange(12).view(3, 4)
print("Original Tensor:\n", base_tensor)

Original Tensor:
 tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])


In [ ]:
# take a slice (this is a view)
my_slice = base_tensor[0:2, 1:3]    #(row, column)
print("sliced view:\n", my_slice)

sliced view:
 tensor([[1, 2],
        [5, 6]])


In [15]:
# mutate the slice
my_slice[0, 0] = 999
print("original tensor after mutating the slice:\n", base_tensor)

original tensor after mutating the slice:
 tensor([[  0, 999,   2,   3],
        [  4,   5,   6,   7],
        [  8,   9,  10,  11]])


In [ ]:
# copies (fancy indexing) are safe
safe_tensor = torch.arange(12).view(3, 4)
print("safe tensor:\n", safe_tensor)

# Fancy index (This is a COPY because we passed a list of rows)
my_copy = safe_tensor[[2, 0]]   # 3rd row and then 1st row
print("\ncopied rows:\n", my_copy)

safe tensor:
 tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])

copied rows:
 tensor([[ 8,  9, 10, 11],
        [ 0,  1,  2,  3]])


In [20]:
# mutate the copy
my_copy[0,0] = 777
print("original safe tensor : ",safe_tensor)
print("\nmy copy tensor : ",my_copy)

original safe tensor :  tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])

my copy tensor :  tensor([[777,   9,  10,  11],
        [  0,   1,   2,   3]])


In [21]:
# boolean masking flattens
tokens = torch.tensor([
    [101, 2045, 102, 0, 0], # 0 is padding
    [101, 888, 999, 102, 0]
])

# create a boolean mask where tokens are NOT padding
pad_mask = (tokens != 0)
print("The Mask:\n", pad_mask)

The Mask:
 tensor([[ True,  True,  True, False, False],
        [ True,  True,  True,  True, False]])


In [22]:
# apply the mask
real_tokens = tokens[pad_mask]
print("Extracted real tokens (notice it is now flat 1D): \n",real_tokens)

Extracted real tokens (notice it is now flat 1D): 
 tensor([ 101, 2045,  102,  101,  888,  999,  102])


In [ ]:
# power tools (None and ...)
# Part A: None (Adding a dimension)
single_sequence = torch.tensor([5, 6, 7, 8])
print("Original shape: ", single_sequence.shape)
print(single_sequence)

# add a fake batch dimension using None
batched_sequence = single_sequence[None, :]
print("\nBatched shape (using None):", batched_sequence.shape)
print(batched_sequence)

Original shape:  torch.Size([4])
tensor([5, 6, 7, 8])

Batched shape (using None): torch.Size([1, 4])
tensor([[5, 6, 7, 8]])


In [26]:
# Part B: ... (Skipping dimensions)
# lets create a 3D tensor: [Batch=2, Sequence=3, Embedding=4]
# think of this as 2 sentences, 3 words each, 4-number embedding per word
complex_tensor = torch.arange(24).view(2,3,4)
print("Complex Tensor shape: ", complex_tensor.shape)
print(complex_tensor)

Complex Tensor shape:  torch.Size([2, 3, 4])
tensor([[[ 0,  1,  2,  3],
         [ 4,  5,  6,  7],
         [ 8,  9, 10, 11]],

        [[12, 13, 14, 15],
         [16, 17, 18, 19],
         [20, 21, 22, 23]]])


In [27]:
# Goal: We want the very first embedding number of Every word in Every sentence.
# The long way: complex_tensor[:,:, 0]
# the power-tool way: complex_tensor[..., 0]

first_elements = complex_tensor[..., 0]
print("Shape after Ellipsis slice: ", first_elements.shape)
print("\nExtracted first elements:\n", first_elements)

Shape after Ellipsis slice:  torch.Size([2, 3])

Extracted first elements:
 tensor([[ 0,  4,  8],
        [12, 16, 20]])


## section 6

In [29]:
# the view crash
t = torch.arange(6).view(2,3)
print("original t : \n",t)

original t : 
 tensor([[0, 1, 2],
        [3, 4, 5]])


In [30]:
# transpose it - this messes up the memory layout
t_transposed = t.t()
print("Transposed t :\n",t_transposed)
print("\nIs contiguous?", t_transposed.is_contiguous())

Transposed t :
 tensor([[0, 3],
        [1, 4],
        [2, 5]])

Is contiguous? False


In [31]:
# try a different view
try:
    # the worker goes on strike
    t_transposed.view(6)
except Exception as e:
    print(f"Caught the crash:\n",e)

Caught the crash:
 view size is not compatible with input tensor's size and stride (at least one dimension spans across two contiguous subspaces). Use .reshape(...) instead.


In [32]:
# fix the crash with contiguous and view
fixed_explicitly = t_transposed.contiguous().view(6)
print("Fixed with contiguous().view(): \n",fixed_explicitly)
print("\nIs fixed explicitly contiguous: ",fixed_explicitly.is_contiguous())

Fixed with contiguous().view(): 
 tensor([0, 3, 1, 4, 2, 5])

Is fixed explicitly contiguous:  True


In [34]:
# fix with reshape - hide the problem
fixed_secretly = t_transposed.reshape(6)
print("Fixed with reshape() : \n",fixed_secretly)
print(f"\nIs fixed secretly contiguoug: {fixed_secretly.is_contiguous()}")

Fixed with reshape() : 
 tensor([0, 3, 1, 4, 2, 5])

Is fixed secretly contiguoug: True


In [36]:
# Power tools (-1 and flatten)
big_tensor = torch.arange(24).view(2,3,4)   # shape : (2, 3, 4)

# i know i want 2 rows but i dont want to calculate 3 * 4. use -1
auto_sized = big_tensor.view(2, -1)
print(f"shape with .view(2,-1) : {auto_sized.shape}")


shape with .view(2,-1) : torch.Size([2, 12])


In [37]:
# flatten everthing from 1 to the end
squished = big_tensor.flatten(start_dim=1)
print(f"shape with .flatten(1) : {squished.shape}")

shape with .flatten(1) : torch.Size([2, 12])


>>The original shape is `(2, 3, 4)`, and in Python, dimensions are numbered starting at 0 (Dim 0 is 2, Dim 1 is 3, Dim 2 is 4).

>>By saying `start_dim=1`, you are telling PyTorch: "Leave Dimension 0 exactly as it is, but take Dimension 1 and everything after it and squish them together." Since Dimensions 1 and 2 are 3 and 4, PyTorch multiplies them (3 × 4 = 12), leaving you with a final shape of `(2, 12)`.

## section 7

In [40]:
# the trap of bare squeeze()
tensor = torch.arange(5).view(1, 5, 1)
print(f"original shape:",{tensor.shape})
print(tensor)

original shape: {torch.Size([1, 5, 1])}
tensor([[[0],
         [1],
         [2],
         [3],
         [4]]])


In [43]:
# only remove the Features dimension
safe_squeeze = tensor.squeeze(2)    
print("safe squeeze(2) shape: ",safe_squeeze.shape)
print(safe_squeeze)

safe squeeze(2) shape:  torch.Size([1, 5])
tensor([[0, 1, 2, 3, 4]])


In [44]:
# remove everything that is size 1
dangerous_squeeze = tensor.squeeze()
print("Dangerous squeeze shape ",dangerous_squeeze.shape)
print(dangerous_squeeze)

Dangerous squeeze shape  torch.Size([5])
tensor([0, 1, 2, 3, 4])


In [45]:
# Expand (Hologram) vs repeat (photocopy)
row = torch.tensor([[1, 2, 3]]) # shape (1, 3)
print("original row : ",row)

original row :  tensor([[1, 2, 3]])


In [46]:
# EXPAND: Free hologram (stride = 0)
expanded = row.expand(4,3)
print("Expanded shape : ",expanded.shape)
print(expanded)

print("\nExpanded strides : ", expanded.stride() )  # <- Look at the 0

Expanded shape :  torch.Size([4, 3])
tensor([[1, 2, 3],
        [1, 2, 3],
        [1, 2, 3],
        [1, 2, 3]])

Expanded strides :  (0, 1)


### Note:

that `0` is the secret to how PyTorch fakes its dimensions!

When you print `expanded.stride()`, you will see an output that looks like this: `(0, 1)`.

Let's bring back the **Warehouse Aisle** analogy to explain exactly what `(0, 1)` means. Remember, a stride tells the warehouse worker: *(Steps to next Row, Steps to next Column)*.

* **The `1` (Column Stride):** This tells the worker, "To move to the next column in the display window, take **1 step** forward down the warehouse aisle." This is perfectly normal. It's how they read `1`, then `2`, then `3`.
* **The `0` (Row Stride) - THE MAGIC TRICK:** This tells the worker, "To move down to the next row in the display window, take **0 steps** in the warehouse."

#### What happens in reality:

1. The worker reads the first row: `1, 2, 3`.
2. PyTorch says, "Okay, now draw the second row."
3. The worker looks at the recipe card, sees the Row Stride is **0**, and says: "Okay, I won't take any steps forward. I will just stand exactly where I started."
4. The worker reads `1, 2, 3` again!

By setting the row stride to `0`, PyTorch forces the computer to re-read the exact same piece of memory over and over again, stacking them visually as "new" rows. It never actually built new rows in the warehouse. That is why `expand()` uses absolutely zero extra memory, and that `0` in the stride output is the mathematical proof of the "hologram" trick!

In [49]:
# REPEAT: Physical photocopy
repeated = row.repeat(4,1)
print("Repeated strides: ",repeated.stride())   # <- Normal strides, real memory
print(repeated)

Repeated strides:  (3, 1)
tensor([[1, 2, 3],
        [1, 2, 3],
        [1, 2, 3],
        [1, 2, 3]])


### Note:

When you print `expanded` and `repeated`, the output looks **exactly identical** on your screen. But that `(3, 1)` stride proves that PyTorch achieved that result in a completely different way.

Here is exactly what happened using our Warehouse analogy.

#### 1. What the Movers Did (The Photocopy)

When you called `repeat(4, 1)`, PyTorch actually allocated brand new memory. It took your original 3 boxes `[1, 2, 3]` and physically photocopied them 4 times into a brand new, long warehouse aisle.

Your new flat memory aisle physically looks like this now:
`[1, 2, 3, 1, 2, 3, 1, 2, 3, 1, 2, 3]` (12 actual boxes taking up memory).

#### 2. What the `(3, 1)` Stride Means

Now, PyTorch assigns a worker to display these 12 boxes as a 4x3 grid. The worker gets a new recipe card with the stride `(3, 1)`:

* **The `1` (Column Stride):** "Take 1 step to get the next number in the row."
* **The `3` (Row Stride):** "To move down to the next row, jump **3 steps** forward in the warehouse."

#### Let's Walk the Warehouse with the Worker:

* **Row 1:** The worker starts at box 0. Reads `1`, takes one step, reads `2`, takes one step, reads `3`.
* **Row 2:** Now the worker needs to start the second row. The recipe says "Jump 3 steps forward from where the last row started." So the worker jumps past the first three boxes and lands on Box 3. What is in Box 3? It's the physical photocopy of `1`. The worker reads `1, 2, 3`.
* **Row 3:** The worker jumps 3 steps forward again, landing on Box 6. Reads the next physical photocopy: `1, 2, 3`.

#### The Summary

* **`expand` (Stride 0):** Used 3 boxes total. To make a new row, the worker jumped **0 steps** and just read the exact same 3 boxes again. (Hologram)
* **`repeat` (Stride 3):** Used 12 boxes total. To make a new row, the worker jumped **3 steps** to reach the next physical set of copied boxes. (Photocopy)

They give the exact same visual result on your screen, but `repeat` actually chewed up your computer's RAM to do it!

In [51]:
# You do this, thinking you are only changing Row 0...
expanded[0, 0] = 999

print("The Expanded Tensor:")
print(expanded) 
# Look at the output! EVERY row now starts with 999!

print("\nThe Original Row:")
print(row)
# The original row was destroyed and changed to 999 too!

The Expanded Tensor:
tensor([[999,   2,   3],
        [999,   2,   3],
        [999,   2,   3],
        [999,   2,   3]])

The Original Row:
tensor([[999,   2,   3]])


In [52]:
# Try to add 1 to the entire expanded tensor
try:
    expanded += 1
except Exception as e:
    print(f"CRASH: {e}")

CRASH: unsupported operation: more than one element of the written-to tensor refers to a single memory location. Please clone() the tensor before performing the operation.


>> Now we get the crash: RuntimeError: unsupported operation: more than one element of the written-to tensor refers to a single memory location.

>> PyTorch panics here because it tries to add 1 to Row 0, and then add 1 to Row 1... but it suddenly realizes it is trying to overwrite the exact same physical warehouse box multiple times simultaneously.

In [55]:
# permute vs transpose
# a 4D tensor: (Batch=2, Channels=3, Height=4, Width=5)
img = torch.arange(120).view(2, 3, 4, 5)
print(f"original image: {img.shape}")
print(img)

original image: torch.Size([2, 3, 4, 5])
tensor([[[[  0,   1,   2,   3,   4],
          [  5,   6,   7,   8,   9],
          [ 10,  11,  12,  13,  14],
          [ 15,  16,  17,  18,  19]],

         [[ 20,  21,  22,  23,  24],
          [ 25,  26,  27,  28,  29],
          [ 30,  31,  32,  33,  34],
          [ 35,  36,  37,  38,  39]],

         [[ 40,  41,  42,  43,  44],
          [ 45,  46,  47,  48,  49],
          [ 50,  51,  52,  53,  54],
          [ 55,  56,  57,  58,  59]]],


        [[[ 60,  61,  62,  63,  64],
          [ 65,  66,  67,  68,  69],
          [ 70,  71,  72,  73,  74],
          [ 75,  76,  77,  78,  79]],

         [[ 80,  81,  82,  83,  84],
          [ 85,  86,  87,  88,  89],
          [ 90,  91,  92,  93,  94],
          [ 95,  96,  97,  98,  99]],

         [[100, 101, 102, 103, 104],
          [105, 106, 107, 108, 109],
          [110, 111, 112, 113, 114],
          [115, 116, 117, 118, 119]]]])


In [56]:
# transpose can only swap two things (e.g. channels and width)
transposed = img.transpose(1, 3)
print(f"Transposed (1, 3): {transposed.shape}")
print(transposed)

Transposed (1, 3): torch.Size([2, 5, 4, 3])
tensor([[[[  0,  20,  40],
          [  5,  25,  45],
          [ 10,  30,  50],
          [ 15,  35,  55]],

         [[  1,  21,  41],
          [  6,  26,  46],
          [ 11,  31,  51],
          [ 16,  36,  56]],

         [[  2,  22,  42],
          [  7,  27,  47],
          [ 12,  32,  52],
          [ 17,  37,  57]],

         [[  3,  23,  43],
          [  8,  28,  48],
          [ 13,  33,  53],
          [ 18,  38,  58]],

         [[  4,  24,  44],
          [  9,  29,  49],
          [ 14,  34,  54],
          [ 19,  39,  59]]],


        [[[ 60,  80, 100],
          [ 65,  85, 105],
          [ 70,  90, 110],
          [ 75,  95, 115]],

         [[ 61,  81, 101],
          [ 66,  86, 106],
          [ 71,  91, 111],
          [ 76,  96, 116]],

         [[ 62,  82, 102],
          [ 67,  87, 107],
          [ 72,  92, 112],
          [ 77,  97, 117]],

         [[ 63,  83, 103],
          [ 68,  88, 108],
          [ 73,  93,

In [57]:
# permute can rearrange everything at once
permuted = img.permute(0, 2, 3, 1)  # move channels (1) to the very end
print(f"Permuted (0, 2, 3, 1) : ", permuted.shape)
print(permuted)

Permuted (0, 2, 3, 1) :  torch.Size([2, 4, 5, 3])
tensor([[[[  0,  20,  40],
          [  1,  21,  41],
          [  2,  22,  42],
          [  3,  23,  43],
          [  4,  24,  44]],

         [[  5,  25,  45],
          [  6,  26,  46],
          [  7,  27,  47],
          [  8,  28,  48],
          [  9,  29,  49]],

         [[ 10,  30,  50],
          [ 11,  31,  51],
          [ 12,  32,  52],
          [ 13,  33,  53],
          [ 14,  34,  54]],

         [[ 15,  35,  55],
          [ 16,  36,  56],
          [ 17,  37,  57],
          [ 18,  38,  58],
          [ 19,  39,  59]]],


        [[[ 60,  80, 100],
          [ 61,  81, 101],
          [ 62,  82, 102],
          [ 63,  83, 103],
          [ 64,  84, 104]],

         [[ 65,  85, 105],
          [ 66,  86, 106],
          [ 67,  87, 107],
          [ 68,  88, 108],
          [ 69,  89, 109]],

         [[ 70,  90, 110],
          [ 71,  91, 111],
          [ 72,  92, 112],
          [ 73,  93, 113],
          [ 74,  9

## section 8

In [ ]:
# broadcasting magic
A = torch.tensor([[10], [20], [30]])    # shape (3, 1)
B = torch.tensor([1, 2, 3]) # shape (3, 1)

print(f"A shape: {A.shape} \n",A)
print(f"\nB shape: {B.shape} \n",B)

# Broadcasting aligns from the right
# A: 3 x 1
# B:     3
# result: 3 x 3
# during broadcasting it becomes like this
#  [[10, 10, 10]    x   []
#   [20, 20, 20]
#   [30, 30, 30]]
C = A + B
print("\nA + B (Broadcasted into a 3x3 matrix) : \n",C)

A shape: torch.Size([3, 1]) 
 tensor([[10],
        [20],
        [30]])

B shape: torch.Size([3]) 
 tensor([1, 2, 3])

A + B (Broadcasted into a 3x3 matrix) : 
 tensor([[11, 12, 13],
        [21, 22, 23],
        [31, 32, 33]])


In [ ]:
# the silent killer (accidental broadcase)
scores = torch.tensor([1.0, 2.0, 3.0, 4.0]) # shape (4,)
baseline = torch.tensor([[1.0], [2.0], [3.0], [4.0]]) # shape (4,1)

print(f"score shape: {scores.shape}, Baseline shape: {baseline.shape}")

print(scores)
print(baseline)

score shape: torch.Size([4]), Baseline shoae: torch.Size([4, 1])
tensor([1., 2., 3., 4.])
tensor([[1.],
        [2.],
        [3.],
        [4.]])


In [66]:
# we want to subtract the baseline from each score
result = scores - baseline
print("accidenta 4x4 output instead of 4 items : \n",result)

accidenta 4x4 output instead of 4 items : 
 tensor([[ 0.,  1.,  2.,  3.],
        [-1.,  0.,  1.,  2.],
        [-2., -1.,  0.,  1.],
        [-3., -2., -1.,  0.]])


In [69]:
# reduction nd keepdim=True
batch = torch.tensor([
    [10, 20, 30],   # setence 1 (sum = 60)
    [1, 2, 3]      # sentence 2 (sum = 6)
], dtype=torch.float32) # shaoe: (2, 3)

print("original batch shaps: ",batch.shape)
print(batch)

original batch shaps:  torch.Size([2, 3])
tensor([[10., 20., 30.],
        [ 1.,  2.,  3.]])


In [70]:
# crush dimensio 1 (the words)
crushed = batch.sum(dim=1)
print(f"sum keepdim=False : {crushed.shape} -> \n{crushed}")

sum keepdim=False : torch.Size([2]) -> 
tensor([60.,  6.])


In [71]:
# crushed dimension 1, but keep the placeholder
crushed_kept = batch.sum(dim=1, keepdim=True)
print(f"sum keepdim=True : {crushed_kept.shape} -> \n{crushed_kept}")

sum keepdim=True : torch.Size([2, 1]) -> 
tensor([[60.],
        [ 6.]])


In [74]:
# Why we did that: Dividing the original tensor by the sum
# batch (2, 3) / crushed (2,)          <-- BROADCASTING CRASHES/FAILS HERE
# batch (2, 3) / crushed_kept (2, 1)   <-- BROADCASTING WORKS PERFECTLY HERE

print("\nnormalized batch using keepdim: \n",batch / crushed_kept)


normalized batch using keepdim: 
 tensor([[0.1667, 0.3333, 0.5000],
        [0.1667, 0.3333, 0.5000]])


In [75]:
print("normalized batch - not using keepdim : \n",batch / crushed)

RuntimeError: The size of tensor a (3) must match the size of tensor b (2) at non-singleton dimension 1

In [77]:
# masked mean pooling (full pipeline)

# 1 batch, 3 words, 2 features per word
embeddings = torch.tensor([
    [
        [1.0, 2.0], # word 1
        [3.0, 4.0], # word 2
        [9.0, 9.0]  # word 3    (This is PADDING, we want to ignore it)
    ]
])

# mask (1=real, 0=padding)
mask = torch.tensor([[1, 1, 0]])

# step 1 and 2 : zero out the padding embeddings (broadcast mask from (1, 3, 1) against (1, 3, 2))
masked_embeddings = embeddings * mask.unsqueeze(-1)
print("zeroed out paddings: \n", masked_embeddings)

zeroed out paddings: 
 tensor([[[1., 2.],
         [3., 4.],
         [0., 0.]]])


In [78]:
# step 3: sum the remaining words together (crush the sequence dim)
summed = masked_embeddings.sum(dim=1)

print(summed)

tensor([[4., 6.]])


In [ ]:
# step 4 - count how many real words we had (crush sequence dim, keep placeholder for division)
# mask shape - (batch, word)
real_word_count = mask.sum(dim=1, keepdim=True)
print(real_word_count)

tensor([[2]])


In [80]:
# step 5 average it out (divide the sum by 2 real words, completely ignoring the 9.0 padding)
sentence_vector = summed / real_word_count
print(f"final sentence vector (average of [1, 2] and [3, 4]) : \n{sentence_vector}")

final sentence vector (average of [1, 2] and [3, 4]) : 
tensor([[2., 3.]])


## Exercise 1 — Token Batch Bootstrap

A tokenizer just handed you a **ragged** batch (sequences of different lengths). Build:

1. `padded` — an `int64` tensor of shape `(batch, max_len)`, short rows filled with `PAD_ID`
2. `pad_mask` — a `bool` tensor of the same shape, `True` where the token is real

**Decision you're practicing:** which dtype each artifact needs (`int64` for IDs because `nn.Embedding` is a table lookup; `bool` for masks) and which creation functions get you there (`torch.full`, `torch.zeros`, `dtype=`).

In [81]:
ragged_token_ids = [
    [5, 12, 7],
    [3, 9],
    [42, 8, 15, 2, 6],
]
PAD_ID = 0

# TODO: build `padded`   — shape (3, 5), dtype int64, short rows filled with PAD_ID
# TODO: build `pad_mask` — shape (3, 5), dtype bool, True at real-token positions
padded = None
pad_mask = None

**Verification** — run after filling the stub.

In [ ]:
# --- Verification: Exercise 1 ---
assert padded is not None and pad_mask is not None, "fill in the stub above first"
assert padded.dtype == torch.long,   f"token IDs must be int64/long, got {padded.dtype}"
assert pad_mask.dtype == torch.bool, f"masks must be bool, got {pad_mask.dtype}"
assert padded.shape == (3, 5) and pad_mask.shape == (3, 5), "shape must be (batch=3, max_len=5)"
assert padded[0].tolist() == [5, 12, 7, 0, 0]
assert padded[1].tolist() == [3, 9, 0, 0, 0]
assert padded[2].tolist() == [42, 8, 15, 2, 6]
assert pad_mask.sum().item() == 3 + 2 + 5, "mask must be True at exactly the real-token positions"

toy_embedding = nn.Embedding(num_embeddings=100, embedding_dim=8, padding_idx=PAD_ID)
token_vectors = toy_embedding(padded)   # this line raises if padded is a float tensor
print(f"nn.Embedding accepted the batch → output shape {tuple(token_vectors.shape)}  # (batch, max_len, embed_dim)")
print("Exercise 1 passed ✓")

## Exercise 2 — Storage Detective

For each derived tensor below, **predict** whether it shares storage with `base`, then implement `shares_storage` to check yourself. (Hint from notes §2: views re-describe the same flat memory; fancy indexing and forced copies allocate new memory. `t.untyped_storage().data_ptr()` identifies the underlying block.)

**Decision you're practicing:** knowing *which operations alias memory* — the difference between a free view and a hidden copy.

In [ ]:
base = torch.arange(12).view(3, 4)

derived = {
    "row_slice":            base[1],
    "transpose":            base.t(),
    "fancy_rows":           base[[2, 0]],
    "reshape_contiguous":   base.reshape(4, 3),
    "reshape_of_transpose": base.t().reshape(12),
    "contiguous_noop":      base.contiguous(),
}

# TODO: replace each None with your True/False prediction (True = shares storage with base)
predictions = {
    "row_slice":            None,
    "transpose":            None,
    "fancy_rows":           None,
    "reshape_contiguous":   None,
    "reshape_of_transpose": None,
    "contiguous_noop":      None,
}

# TODO: implement using untyped_storage().data_ptr()
def shares_storage(tensor_a, tensor_b):
    """Return True if the two tensors are backed by the same memory block."""
    pass

**Verification**

In [ ]:
# --- Verification: Exercise 2 ---
rules = {
    "row_slice":            "basic slicing is a view",
    "transpose":            "transpose only swaps strides",
    "fancy_rows":           "fancy indexing must gather scattered elements → copy",
    "reshape_contiguous":   "reshape on a contiguous tensor degrades to a view",
    "reshape_of_transpose": "reshape on non-contiguous memory must copy",
    "contiguous_noop":      "contiguous() on an already-contiguous tensor returns self",
}

def _shares(a, b):
    return a.untyped_storage().data_ptr() == b.untyped_storage().data_ptr()

wrong = 0
for name, tensor in derived.items():
    truth = _shares(tensor, base)
    assert predictions[name] is not None, f"no prediction for {name!r}"
    assert shares_storage(tensor, base) == truth, "your shares_storage() disagrees with data_ptr ground truth"
    mark = "✓" if predictions[name] == truth else "✗"
    wrong += predictions[name] != truth
    print(f"{mark} {name:22s} shares={str(truth):5s} — {rules[name]}")
assert wrong == 0, f"{wrong} prediction(s) wrong — revisit notes §2, §5, §6"
print("Exercise 2 passed ✓")

## Exercise 3 — Fix the `view()` Crash

The classic multi-head-attention merge bug. `context` is `(batch, heads, seq, head_dim)`; after `transpose(1, 2)` the tensor is **non-contiguous**, and this line explodes:

```python
context.transpose(1, 2).view(BATCH, SEQ_LEN, HEADS * HEAD_DIM)   # 💥 RuntimeError
```

Write **two** fixes and, in a comment, explain in stride terms why the original crashes:

- `merge_heads_explicit` — using `.contiguous().view(...)` (the copy is *visible*)
- `merge_heads_reshape` — using `.reshape(...)` (the copy happens *silently*)

**Decision you're practicing:** `view` (loud no-copy guarantee) vs `reshape` (silent maybe-copy) vs `.contiguous().view()` (explicit copy).

In [ ]:
BATCH, HEADS, SEQ_LEN, HEAD_DIM = 2, 3, 4, 5
context = torch.randn(BATCH, HEADS, SEQ_LEN, HEAD_DIM)   # shape: (batch, heads, seq, head_dim)

def merge_heads_explicit(context_tensor):
    """(batch, heads, seq, head_dim) -> (batch, seq, heads*head_dim) via contiguous().view()."""
    # TODO
    pass

def merge_heads_reshape(context_tensor):
    """(batch, heads, seq, head_dim) -> (batch, seq, heads*head_dim) via reshape()."""
    # TODO
    pass

**Verification**

In [ ]:
# --- Verification: Exercise 3 ---
reference = context.permute(0, 2, 1, 3).contiguous().view(BATCH, SEQ_LEN, HEADS * HEAD_DIM)

crashed = False
try:
    context.transpose(1, 2).view(BATCH, SEQ_LEN, HEADS * HEAD_DIM)
except RuntimeError as err:
    crashed = True
    print(f"original still crashes, as expected:\n  RuntimeError: {err}\n")
assert crashed, "the buggy line should raise — did context get replaced by a contiguous copy?"

for fix in (merge_heads_explicit, merge_heads_reshape):
    merged = fix(context)
    assert merged is not None, f"{fix.__name__} not implemented yet"
    assert merged.shape == (BATCH, SEQ_LEN, HEADS * HEAD_DIM), f"{fix.__name__}: wrong shape {tuple(merged.shape)}"
    assert torch.allclose(merged, reference), f"{fix.__name__}: values scrambled — merged the wrong dims?"
    print(f"{fix.__name__:22s} ✓ matches reference")
print("Exercise 3 passed ✓")

## Exercise 4 — Zero-Copy Mask Broadcast

Attention scores are `(batch, heads, seq_q, seq_k)`; your padding mask is `(batch, seq)` and says which **key** positions are real. Write `broadcast_pad_mask(mask, num_heads)` returning a `(batch, heads, seq, seq)` tensor using **only `unsqueeze` and `expand`** — zero bytes copied.

**Decision you're practicing:** `expand` (free, stride-0, read-only) vs `repeat` (real copy) — and *proving* zero-copy with `.stride()` and storage pointers.

In [ ]:
pad_mask = torch.tensor([
    [True, True, True,  False],
    [True, True, False, False],
])                                   # shape: (batch=2, seq=4) — True = real token

def broadcast_pad_mask(mask, num_heads):
    """(batch, seq) -> (batch, num_heads, seq, seq) using ONLY unsqueeze + expand."""
    # TODO
    pass

NUM_HEADS = 3
attn_mask = broadcast_pad_mask(pad_mask, NUM_HEADS)

**Verification**

In [ ]:
# --- Verification: Exercise 4 ---
assert attn_mask is not None, "fill in the stub above first"
assert attn_mask.shape == (2, NUM_HEADS, 4, 4), f"wrong shape: {tuple(attn_mask.shape)}"
assert 0 in attn_mask.stride(), "no stride-0 dims → you copied (repeat?) instead of expanding"
assert attn_mask.untyped_storage().data_ptr() == pad_mask.untyped_storage().data_ptr(), \
    "result must alias the original mask's storage (views only!)"

reference = pad_mask[:, None, None, :].repeat(1, NUM_HEADS, 4, 1)   # the memory-hungry way
assert torch.equal(attn_mask, reference), "values differ from the repeat-based reference"
print(f"zero-copy confirmed: result strides = {attn_mask.stride()}")
print(f"memory check: expand added 0 bytes; repeat would have allocated {reference.numel()} bools")
print("Exercise 4 passed ✓")

## Exercise 5 — Broadcast or Bust

For each shape pair, predict the broadcast **result shape** (a tuple) or the string `"error"`. Pair 3 (index starts at 0) is the silent killer from notes §8 — look twice.

**Decision you're practicing:** running the two right-alignment rules in your head *before* PyTorch runs them for you.

In [ ]:
shape_pairs = [
    ((4, 1),    (3,)),
    ((2, 3, 4), (3, 4)),
    ((2, 3, 4), (4, 3)),
    ((5,),      (5, 1)),
    ((1,),      (3, 4)),
    ((2, 1, 4), (1, 3, 1)),
    ((3, 2),    (2, 3)),
    ((6, 1, 8), (8,)),
]

# TODO: replace each None with a shape tuple like (2, 3, 4), or the string "error"
predicted_shapes = [None] * 8

**Verification**

In [ ]:
# --- Verification: Exercise 5 ---
wrong = 0
for pair_index, (shape_a, shape_b) in enumerate(shape_pairs):
    try:
        actual = tuple(torch.broadcast_shapes(shape_a, shape_b))
    except RuntimeError:
        actual = "error"
    prediction = predicted_shapes[pair_index]
    assert prediction is not None, f"no prediction for pair {pair_index}"
    mark = "✓" if prediction == actual else "✗"
    wrong += prediction != actual
    print(f"{mark} pair {pair_index}: {str(shape_a):12s} vs {str(shape_b):12s} → {str(actual):12s} (you said {prediction})")
if wrong:
    print("\nRe-run the rules: right-align; equal sizes pass; a 1 (or missing dim) stretches; anything else errors.")
assert wrong == 0, f"{wrong} prediction(s) wrong"
print("Exercise 5 passed ✓")

## Exercise 6 — Masked Mean Pooling

Turn per-token embeddings `(batch, seq, dim)` into one sentence vector `(batch, dim)` by averaging **only the real tokens** — no Python loops. Row 1 of the test batch has a single real token, so an off-by-`keepdim` bug shows up immediately.

**Decision you're practicing:** broadcasting + reductions with `keepdim=True` — the full shape choreography of notes §8.

In [ ]:
def masked_mean_pool(token_embeddings, mask):
    """Average real-token embeddings per sequence.

    Args:
        token_embeddings: float tensor, shape (batch, seq, dim)
        mask: bool tensor, shape (batch, seq) — True where the token is real
    Returns:
        (batch, dim) tensor: per-sequence mean over real tokens only.
    Math note: pooled_b = sum_t(emb_bt * mask_bt) / sum_t(mask_bt)
    """
    # TODO — no loops!
    pass

**Verification**

In [ ]:
# --- Verification: Exercise 6 ---
torch.manual_seed(0)
test_embeddings = torch.randn(3, 4, 2)          # shape: (batch=3, seq=4, dim=2)
test_mask = torch.tensor([
    [True, True,  True,  False],
    [True, False, False, False],                # single real token — punishes keepdim mistakes
    [True, True,  False, False],
])

pooled = masked_mean_pool(test_embeddings, test_mask)
assert pooled is not None, "fill in the stub above first"
assert pooled.shape == (3, 2), f"expected (3, 2), got {tuple(pooled.shape)}"

# loop reference: the definition, written the slow obvious way
for row in range(3):
    real_vectors = test_embeddings[row][test_mask[row]]     # (n_real, dim) via boolean indexing
    expected = real_vectors.mean(dim=0)
    assert torch.allclose(pooled[row], expected, atol=1e-6), f"row {row}: {pooled[row]} vs expected {expected}"
    print(f"row {row}: {int(test_mask[row].sum())} real token(s) → pooled {[round(v, 4) for v in pooled[row].tolist()]} ✓")
print("Exercise 6 passed ✓")

## Exercise 7 — dtype Triage

Three tensors, three jobs. Pick the right dtype for each, apply the cast, and record your choice in `dtype_choices`:

1. `big_scores` — raw attention logits reaching 70,000; must **survive a 16-bit cast**
2. `float_token_ids` — token IDs that arrived as floats; must **feed `nn.Embedding`**
3. `tiny_bump` — the value 1.001, where the +0.001 must **survive a 16-bit cast**

**Decision you're practicing:** the dtype table from notes §4 — fp16's *range* problem vs bf16's *precision* problem, and why IDs are `int64`.

In [ ]:
big_scores      = torch.tensor([70000.0, -12.5, 300.25])   # shape: (3,)
float_token_ids = torch.tensor([[3.0, 17.0, 0.0]])         # shape: (1, 3)
tiny_bump       = torch.tensor(1.001)                      # scalar

# TODO: record your choices — values must be torch dtypes (e.g. torch.bfloat16)
dtype_choices = {
    "big_scores":      None,
    "float_token_ids": None,
    "tiny_bump":       None,
}

# TODO: apply the casts
big_scores_cast      = None
float_token_ids_cast = None
tiny_bump_cast       = None

**Verification**

In [ ]:
# --- Verification: Exercise 7 ---
assert torch.isinf(big_scores.to(torch.float16)).any(), "sanity: fp16 must overflow on 70000"
assert big_scores_cast is not None, "fill in the stub above first"
assert big_scores_cast.dtype == torch.bfloat16, "big scores need bf16 — fp32 range in 16 bits"
assert torch.isfinite(big_scores_cast.float()).all(), "your cast overflowed!"

assert float_token_ids_cast.dtype == torch.long, "embedding indices must be int64/long"
_ = nn.Embedding(50, 4)(float_token_ids_cast)     # raises on float dtypes
print("nn.Embedding accepted the cast IDs ✓")

assert tiny_bump_cast.dtype == torch.float16, "only fp16 has fine enough spacing at 1.0 to keep +0.001"
assert tiny_bump_cast.item() != 1.0, "the bump vanished — check the 16-bit spacing table in notes §4"
assert tiny_bump.to(torch.bfloat16).item() == 1.0   # sanity: bf16 really does round it away

for key, chosen_dtype in dtype_choices.items():
    print(f"choice: {key:16s} → {chosen_dtype}")
print("Exercise 7 passed ✓")

---
## Done!

Compare your work against `solved/ch01-tensors-and-shape-algebra-solved.ipynb`, then move on to **ch02 — Tensor Operations for NLP**. The master decision table at the end of the ch01 notes is worth a re-read now that every row has bitten you at least once.